# 02 · PyTorch: repaso desde la base (≈2 h)

Lo que la oferta llama "experiencia entrenando, ajustando y evaluando modelos de Deep Learning".
Conviene saber **escribir un bucle de entrenamiento de memoria** o explicar qué hace cada
línea. Al final de este notebook deberías poder hacerlo sin mirar.

Índice:
1. Tensores (shape, dtype, device, broadcasting, permute/view)
2. Autograd: cómo calcula PyTorch los gradientes
3. `nn.Module`, `Dataset`, `DataLoader`
4. El bucle de entrenamiento canónico ✍️
5. CNNs: convolución, tamaño de salida ✍️, BatchNorm, residuales
6. Transfer learning (ResNet preentrenada)
7. Pérdidas típicas en visión: CE, BCE, *focal loss* ✍️
8. Trucos de entrenamiento: LR, schedulers, AMP, checkpoints, overfitting

In [ ]:
# ⚙️ SETUP · ejecuta esta celda primero. Hace el notebook independiente: funciona en local (dentro del repo)
# y en Google Colab (instala lo que falte y descarga los datos a ./data).
import sys, subprocess, importlib.util, urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
PIP = {}   # paquete pip → módulo que importa
missing = [p for p, m in PIP.items() if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# Raíz del repo: la carpeta que contiene vision_utils/ (subiendo desde aquí, o el clon de Colab)
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents, Path.cwd() / "vision-bootcamp"]
             if (p / "vision_utils").is_dir()), None)
ROOT = ROOT or Path.cwd()   # fuera del repo (Colab): los datos se guardan en ./data
DATA = ROOT / "data"
DATA.mkdir(exist_ok=True)
# Los pesos YOLO (yolo11n.pt…) no hace falta descargarlos: Ultralytics los baja solo la primera vez que se usan.
print("ROOT:", ROOT, "| DATA:", DATA, "| Colab:", IN_COLAB)

In [ ]:
import time, math
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
torch.manual_seed(0)
print(DEVICE)

## 1. Tensores

Un tensor es un array N-dimensional (como NumPy) con dos extras: **puede vivir en GPU** y **registra
operaciones para calcular gradientes**.

Convenciones de visión en PyTorch:
- Imagen: `(C, H, W)`; batch: `(N, C, H, W)` ("NCHW"). OpenCV/NumPy usan `(H, W, C)`.
- `float32` para entrenar; `float16`/`bfloat16` para acelerar (AMP); `uint8` para imágenes crudas.

**Broadcasting**: dimensiones se alinean por la derecha; una dimensión de tamaño 1 se "estira".
Es la clave para vectorizar (p. ej. calcular IoU de N cajas contra M cajas sin bucles, en el notebook 03).

In [ ]:
x = torch.randn(8, 3, 64, 64)                 # batch de 8 imágenes RGB 64×64
print(x.shape, x.dtype, x.device)

mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)   # media ImageNet por canal
std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
x_norm = (x - mean) / std                       # broadcasting (1,3,1,1) contra (8,3,64,64)

hwc = x[0].permute(1, 2, 0)                     # CHW → HWC (para mostrar)
print("permute:", hwc.shape, "| contiguo:", hwc.is_contiguous())
flat = x.view(8, -1)                            # view requiere memoria contigua; reshape copia si hace falta
print("flatten:", flat.shape)

a = torch.randn(5, 1, 2)   # 5 "cajas"
b = torch.randn(1, 7, 2)   # 7 "cajas"
print("broadcast (5,1,2) - (1,7,2) →", (a - b).shape, " ← todas las parejas sin bucles")

x_gpu = x.to(DEVICE)       # mover a GPU; ¡modelo y datos deben estar en el mismo device!
print(x_gpu.device)

## 2. Autograd

Cada operación sobre tensores con `requires_grad=True` se apunta en un **grafo computacional**.
`loss.backward()` recorre ese grafo hacia atrás aplicando la regla de la cadena y deja en `.grad`
la derivada de la pérdida respecto a cada parámetro. El optimizador luego hace `p -= lr * p.grad`.

Ejemplo: regresión lineal `y = 3x + 2` aprendida "a mano", sin `nn` ni `optim`:

In [ ]:
X = torch.linspace(-1, 1, 100).unsqueeze(1)
Y = 3 * X + 2 + 0.1 * torch.randn_like(X)

w = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
lr = 0.5
for step in range(100):
    pred = X * w + b
    loss = ((pred - Y) ** 2).mean()
    loss.backward()                 # calcula dL/dw y dL/db
    with torch.no_grad():           # la actualización NO debe entrar en el grafo
        w -= lr * w.grad
        b -= lr * b.grad
    w.grad.zero_(); b.grad.zero_()  # ¡los gradientes se ACUMULAN si no los pones a cero!
print(f"w={w.item():.3f} (3), b={b.item():.3f} (2), loss={loss.item():.4f}")

## 3. `nn.Module`, `Dataset` y `DataLoader`

- `nn.Module`: define capas en `__init__` y el cálculo en `forward`. `model.parameters()` da los pesos.
- `Dataset`: implementa `__len__` y `__getitem__(i)` → `(imagen, etiqueta)`.
- `DataLoader`: hace batches, baraja (`shuffle=True` solo en train) y carga en paralelo (`num_workers`).

Usamos **CIFAR-10** (60k imágenes 32×32, 10 clases; ~150 MB la primera vez).
ImageFolder ordena las imágenes por clase → **hay que barajar** al hacer el subconjunto (si no, solo verías 2 clases).

In [ ]:
import torchvision
from torchvision.transforms import v2 as T

train_tf = T.Compose([
    T.ToImage(), T.RandomHorizontalFlip(), T.RandomCrop(32, padding=4),   # augmentations SOLO en train
    T.ToDtype(torch.float32, scale=True), T.Normalize([0.49, 0.48, 0.45], [0.25, 0.24, 0.26]),
])
test_tf = T.Compose([
    T.ToImage(), T.ToDtype(torch.float32, scale=True), T.Normalize([0.49, 0.48, 0.45], [0.25, 0.24, 0.26]),
])
# Descargamos CIFAR-10 como carpetas de imágenes (mirror rápido en GitHub). Formato "ImageFolder":
#   cifar10/train/<clase>/*.png  y  cifar10/test/<clase>/*.png  ← el mismo que usarías con tus propias imágenes
from torchvision.datasets.utils import download_and_extract_archive
if not (DATA / "cifar10").exists():
    download_and_extract_archive("https://github.com/ultralytics/assets/releases/download/v0.0.0/cifar10.zip", DATA)
train_ds = torchvision.datasets.ImageFolder(DATA / "cifar10" / "train", transform=train_tf)
test_ds = torchvision.datasets.ImageFolder(DATA / "cifar10" / "test", transform=test_tf)

# subconjunto para que entrene en ~1 min en el Mac
g = torch.Generator().manual_seed(0)
train_sub = torch.utils.data.Subset(train_ds, torch.randperm(len(train_ds), generator=g)[:20000].tolist())
test_sub = torch.utils.data.Subset(test_ds, torch.randperm(len(test_ds), generator=g)[:2000].tolist())
train_dl = torch.utils.data.DataLoader(train_sub, batch_size=128, shuffle=True, num_workers=0)
test_dl = torch.utils.data.DataLoader(test_sub, batch_size=256, shuffle=False, num_workers=0)

imgs, labels = next(iter(train_dl))
print(imgs.shape, labels[:10], train_ds.classes)

## 4. ✍️ EJERCICIO: el bucle de entrenamiento canónico

Escribe `train_one_epoch(model, dl, opt, loss_fn)` y `evaluate(model, dl)` (devuelve accuracy).
Los 5 pasos sagrados dentro del bucle: **mover a device → forward → loss → `zero_grad` + `backward` → `step`**.
Y no olvides `model.train()` / `model.eval()` y `torch.no_grad()` (o `torch.inference_mode()`) al evaluar.

In [ ]:
def train_one_epoch(model, dl, opt, loss_fn):
    # TU CÓDIGO AQUÍ → devuelve la pérdida media
    raise NotImplementedError

def evaluate(model, dl):
    # TU CÓDIGO AQUÍ → devuelve accuracy en [0, 1]
    raise NotImplementedError

In [ ]:
# SOLUCIÓN
def train_one_epoch(model, dl, opt, loss_fn):
    model.train()                              # activa dropout y BN en modo entrenamiento
    total, n = 0.0, 0
    for x, y in dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)                      # forward
        loss = loss_fn(logits, y)
        opt.zero_grad(set_to_none=True)        # borrar gradientes del paso anterior
        loss.backward()                        # backprop
        opt.step()                             # actualizar pesos
        total += loss.item() * len(x); n += len(x)
    return total / n

@torch.inference_mode()                        # sin grafo → menos memoria y más rápido
def evaluate(model, dl):
    model.eval()                               # BN usa estadísticas acumuladas, dropout off
    correct, n = 0, 0
    for x, y in dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        correct += (model(x).argmax(1) == y).sum().item(); n += len(x)
    return correct / n

## 5. CNNs

**Tamaño de salida de una convolución** (pregunta clásica):

$$ H_{out} = \left\lfloor \frac{H_{in} + 2p - k}{s} \right\rfloor + 1 $$

con kernel `k`, padding `p`, stride `s`. Ej.: 640 con k=3, s=2, p=1 → 320. Los *backbones* de YOLO bajan la
resolución con strides hasta P3/P4/P5 = 1/8, 1/16, 1/32 (para 640: mapas de 80, 40 y 20).

Piezas de una CNN moderna:
- **Conv → BatchNorm → activación** (ReLU / SiLU). BN normaliza por canal usando estadísticas del batch:
  estabiliza y acelera el entrenamiento. ¡En `eval()` usa medias acumuladas! (bug típico: olvidar `model.eval()`).
- **Bloques residuales** (ResNet): `y = x + F(x)`. El gradiente fluye por el atajo → se pueden entrenar redes
  muy profundas.
- **Campo receptivo**: cuánta imagen "ve" cada neurona. Crece con profundidad y stride. Objetos grandes
  necesitan campo receptivo grande → por eso YOLO detecta objetos grandes en P5 y pequeños en P3.

✍️ **Ejercicio rápido**: calcula a mano el tamaño de salida de cada capa de `SmallCNN` para entrada 32×32 y
compruébalo con el print.

In [ ]:
def conv_out(h, k, s=1, p=0):
    return (h + 2 * p - k) // s + 1

print("640, k3 s2 p1 →", conv_out(640, 3, 2, 1))

class SmallCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        def block(cin, cout):
            return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                                 nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                                 nn.MaxPool2d(2))
        self.features = nn.Sequential(block(3, 32), block(32, 64), block(64, 128))   # 32 → 16 → 8 → 4
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.2), nn.Linear(128, n_classes))

    def forward(self, x):
        return self.head(self.features(x))

model = SmallCNN().to(DEVICE)
x = torch.randn(2, 3, 32, 32, device=DEVICE)
for i, layer in enumerate(model.features):
    x = layer(x); print(f"bloque {i}: {tuple(x.shape)}")
print("parámetros:", sum(p.numel() for p in model.parameters()) / 1e3, "k")

In [ ]:
model = SmallCNN().to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=5e-4)
EPOCHS = 4
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-3, total_steps=EPOCHS * len(train_dl))
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.05)

# versión con scheduler por paso (lo normal con OneCycle / cosine)
def train_one_epoch_sched(model, dl, opt, loss_fn, sched):
    model.train(); total, n = 0.0, 0
    for x, y in dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        loss = loss_fn(model(x), y)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
        total += loss.item() * len(x); n += len(x)
    return total / n

hist = []
for ep in range(EPOCHS):
    t0 = time.perf_counter()
    tr = train_one_epoch_sched(model, train_dl, opt, loss_fn, sched)
    acc = evaluate(model, test_dl)
    hist.append((tr, acc))
    print(f"época {ep}: loss={tr:.3f}  test_acc={acc:.3f}  ({time.perf_counter()-t0:.1f}s)")

## 6. Transfer learning

En la vida real casi **nunca entrenas desde cero**: partes de un modelo preentrenado (ImageNet, COCO) y lo
adaptas (*fine-tuning*). Estrategias:
1. **Congelar el backbone** y entrenar solo la cabeza → pocos datos, rápido.
2. **Descongelar todo** con LR pequeño (y quizá LR menor en el backbone) → más datos, mejor resultado.

Esto es exactamente lo que haces con YOLO: `YOLO("yolo11n.pt").train(data=...)` parte de pesos COCO.

Aquí: ResNet18 preentrenada en ImageNet, cambiamos la última capa a 10 clases. Las imágenes de CIFAR son
32×32 pero ResNet espera ~224; las subimos a 96 (compromiso velocidad/precisión).

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

net = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
print("cabeza original:", net.fc)
for p in net.parameters():
    p.requires_grad = False                       # congelar todo
net.fc = nn.Linear(net.fc.in_features, 10)        # nueva cabeza (entrenable por defecto)
net = net.to(DEVICE)

class Upsample(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): return self.m(F.interpolate(x, size=96, mode="bilinear", align_corners=False))

tl_model = Upsample(net)
opt = torch.optim.AdamW([p for p in tl_model.parameters() if p.requires_grad], lr=1e-3)
for ep in range(2):
    tr = train_one_epoch(tl_model, train_dl, opt, nn.CrossEntropyLoss())
    print(f"[solo cabeza] época {ep}: loss={tr:.3f}  test_acc={evaluate(tl_model, test_dl):.3f}")

# fase 2: descongelar el último bloque con LR pequeño
for p in net.layer4.parameters():
    p.requires_grad = True
opt = torch.optim.AdamW([{"params": net.layer4.parameters(), "lr": 1e-4},
                         {"params": net.fc.parameters(), "lr": 1e-3}])
tr = train_one_epoch(tl_model, train_dl, opt, nn.CrossEntropyLoss())
print(f"[layer4 + cabeza] loss={tr:.3f}  test_acc={evaluate(tl_model, test_dl):.3f}")

## 7. Pérdidas típicas en visión

| Pérdida | Dónde | Nota |
|---|---|---|
| `CrossEntropyLoss` | clasificación multiclase | recibe **logits** (no softmax); incluye log-softmax |
| `BCEWithLogitsLoss` | multi-etiqueta, objectness, clases en YOLOv8 | sigmoide por clase, numéricamente estable |
| **Focal loss** | detección (RetinaNet, DETR) | baja el peso de ejemplos fáciles: hay miles de "fondo" por cada objeto |
| L1 / Smooth L1 | regresión de cajas | |
| **IoU / GIoU / CIoU loss** | regresión de cajas (YOLO) | optimiza directamente el solapamiento |
| DFL (Distribution Focal Loss) | YOLOv8/11 | predice cada lado de la caja como una distribución discreta |
| Triplet / contrastiva | ReID (notebook 07) | acerca embeddings de la misma identidad |

**Focal loss**: $FL(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)$, con $p_t$ = probabilidad asignada a la clase
correcta. Si $p_t \to 1$ (ejemplo fácil), $(1-p_t)^\gamma \to 0$ y casi no contribuye. γ=2, α=0.25 típicos.

✍️ **Ejercicio**: implementa `focal_loss(logits, targets, alpha=0.25, gamma=2)` binaria (targets 0/1) y
comprueba que con γ=0 y α=None coincide con BCE.

In [ ]:
def focal_loss(logits, targets, alpha=0.25, gamma=2.0):
    # TU CÓDIGO AQUÍ
    raise NotImplementedError

In [ ]:
# SOLUCIÓN
def focal_loss(logits, targets, alpha=0.25, gamma=2.0):
    p = torch.sigmoid(logits)
    ce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    p_t = p * targets + (1 - p) * (1 - targets)
    loss = ce * (1 - p_t) ** gamma
    if alpha is not None:
        loss = loss * (alpha * targets + (1 - alpha) * (1 - targets))
    return loss.mean()

In [ ]:
lg, tg = torch.randn(1000), (torch.rand(1000) > 0.9).float()
assert torch.allclose(focal_loss(lg, tg, alpha=None, gamma=0), F.binary_cross_entropy_with_logits(lg, tg))
print("✅ focal(γ=0) == BCE  |  focal(γ=2) =", focal_loss(lg, tg).item(), " BCE =", F.binary_cross_entropy_with_logits(lg, tg).item())

## 8. Trucos de entrenamiento que debes saber nombrar

- **Learning rate**: el hiperparámetro más importante. Demasiado alto → diverge (loss NaN); bajo → lento.
  **Warmup** (subir LR los primeros pasos) + **cosine decay** es el estándar (YOLO lo usa).
- **AdamW** (Adam con weight decay desacoplado) vs **SGD+momentum** (YOLO clásico usa SGD; Ultralytics elige
  automáticamente con `optimizer="auto"`).
- **AMP / mixed precision**: forward en FP16/BF16 → ~2× velocidad y mitad de memoria en GPU NVIDIA:
  ```python
  scaler = torch.amp.GradScaler()
  with torch.autocast(device_type="cuda", dtype=torch.float16):
      loss = loss_fn(model(x), y)
  scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
  ```
- **EMA de pesos** (YOLO): mantiene una media móvil de los pesos → modelo final más estable.
- **Early stopping** y **checkpoints**: guarda `state_dict` (no el objeto entero):
  `torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "epoch": ep}, "ckpt.pt")`.
- **Overfitting** (train ↓, val ↑): más datos/augmentations, weight decay, dropout, modelo más pequeño,
  early stopping. **Underfitting** (ambas altas): modelo más grande, más épocas, LR adecuado.
- **Reproducibilidad**: semillas (`torch.manual_seed`), `cudnn.deterministic`, versionar datos y config.
- **Gradient accumulation**: si no cabe un batch grande, acumula gradientes de k mini-batches antes de `step()`.

In [ ]:
ckpt = DATA / "smallcnn.pt"
torch.save({"model": model.state_dict()}, ckpt)
m2 = SmallCNN().to(DEVICE)
m2.load_state_dict(torch.load(ckpt, map_location=DEVICE)["model"])
print("recargado, acc =", evaluate(m2, test_dl))